In [1]:
!pip install langchain-openai --q


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from typing import Literal, Optional
from typing_extensions import TypedDict

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

import warnings
warnings.filterwarnings("ignore")

c:\study\AI\IITM_Agentic_AI_Training\venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
import utils

In [4]:
# ==========================================
# 1. Define Pydantic Models for Structured Output
# ==========================================

class EmailAnalysis(BaseModel):
    """Schema for detecting language, translating, and classifying intent."""
    detected_language: str = Field(description="The original language of the email")
    english_translation: str = Field(description="English translation of the email (or original text if already English)")
    intent: Literal["Praise", "Concern"] = Field(description="Intent classification: Praise or Concern")

class PraiseResponse(BaseModel):
    """Schema for drafting a thank you email."""
    response_draft: str = Field(description="A warm, professional response email referencing the specific praise points")

class ConcernSummary(BaseModel):
    """Schema for extracting key concerns."""
    summary: str = Field(description="A concise bullet-point summary of the key issue(s) raised in the email")

In [5]:
# ==========================================
# 2. Define Graph State
# ==========================================

class EmailAgentState(TypedDict):
    raw_email: str
    detected_language: Optional[str]
    english_translation: Optional[str]
    intent: Optional[Literal["Praise", "Concern"]]
    response_draft: Optional[str]
    summary: Optional[str]


# Initialize LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2)

In [6]:
# ==========================================
# 3. Define Graph Nodes
# ==========================================

def analyze_email_node(state: EmailAgentState) -> dict:
    """Detects language, translates if necessary, and classifies intent."""
    structured_llm = llm.with_structured_output(EmailAnalysis)

    prompt = f"""
    Analyze the following customer email:
    1. Detect its original language.
    2. Translate it to English if it's in another language.
    3. Classify its primary intent as either 'Praise' or 'Concern'.

    Email:
    {state['raw_email']}
    """

    res: EmailAnalysis = structured_llm.invoke([HumanMessage(content=prompt)])

    return {
        "detected_language": res.detected_language,
        "english_translation": res.english_translation,
        "intent": res.intent
    }


def draft_praise_response_node(state: EmailAgentState) -> dict:
    """Drafts a thank you email if the intent is Praise."""
    structured_llm = llm.with_structured_output(PraiseResponse)

    prompt = f"""
    You are a customer support agent. Draft a warm, professional thank-you email response
    referencing the customer's specific positive feedback points below.

    Email (English):
    {state['english_translation']}
    """

    res: PraiseResponse = structured_llm.invoke([HumanMessage(content=prompt)])
    return {"response_draft": res.response_draft}


def summarize_concern_node(state: EmailAgentState) -> dict:
    """Summarizes key points if the intent is Concern."""
    structured_llm = llm.with_structured_output(ConcernSummary)

    prompt = f"""
    Extract a clear, summary of the key issue(s), order info, and requested resolution
    from the customer concern below.

    Email (English):
    {state['english_translation']}
    """

    res: ConcernSummary = structured_llm.invoke([HumanMessage(content=prompt)])
    return {"summary": res.summary}

In [7]:
# ==========================================
# 4. Define Conditional Edge Router
# ==========================================

def route_by_intent(state: EmailAgentState) -> Literal["draft_praise_response", "summarize_concern"]:
    """Routes execution based on the classified intent."""
    if state["intent"] == "Praise":
        return "draft_praise_response"
    return "summarize_concern"

In [8]:
# ==========================================
# 5. Build and Compile the Graph
# ==========================================

workflow = StateGraph(EmailAgentState)

# Add Nodes
workflow.add_node("analyze_email", analyze_email_node)
workflow.add_node("draft_praise_response", draft_praise_response_node)
workflow.add_node("summarize_concern", summarize_concern_node)

# Add Edges
workflow.add_edge(START, "analyze_email")

# Add Conditional Edge from analysis node
workflow.add_conditional_edges(
    "analyze_email",
    route_by_intent,
    {
        "draft_praise_response": "draft_praise_response",
        "summarize_concern": "summarize_concern"
    }
)

# Connect worker nodes to END
workflow.add_edge("draft_praise_response", END)
workflow.add_edge("summarize_concern", END)

# Compile Agent
email_agent = workflow.compile()

In [9]:
praise_email = """Bonjour l'équipe,
Je voulais juste dire que votre service client est fantastique ! Marie m'a aidé à résoudre mon problème de livraison en moins de 10 minutes. Vos produits sont également d'une excellente qualité. Merci beaucoup !
"""

In [10]:
res1 = email_agent.invoke({"raw_email": praise_email})
print(f"Detected Language: {res1['detected_language']}")
print(f"Intent: {res1['intent']}")
print(f"Intent: {res1['english_translation']}")
print(f"Draft Response:\n{res1['response_draft']}\n")

Detected Language: French
Intent: Praise
Intent: Hello team, I just wanted to say that your customer service is fantastic! Marie helped me resolve my delivery issue in less than 10 minutes. Your products are also of excellent quality. Thank you very much!
Draft Response:
Dear customer, Thank you for your kind words and positive feedback! We are thrilled to hear that you had a fantastic experience with our customer service team, especially with Marie resolving your delivery issue promptly. We take pride in providing excellent quality products and exceptional service to our customers. Your appreciation means a lot to us. If you have any further questions or need assistance, please feel free to reach out. Thank you for choosing us! Best regards, [Your Name]



In [11]:
concern_email = """
Hola, Hice un pedido hace más de dos semanas (Número de orden #84920)
y todavía no he recibido ninguna actualización del envío. Además, el importe ya fue cobrado de mi tarjeta.
Intenté llamar pero nadie responde.
Por favor necesito una solución inmediata o el reembolso."""

In [12]:
res2 = email_agent.invoke({"raw_email": concern_email})

In [13]:
print(f"Detected Language: {res2['detected_language']}")
print(f"Intent: {res2['intent']}")
print(f"Intent: {res2['english_translation']}")
print(f"Summary:\n{res2['summary']}")

Detected Language: Spanish
Intent: Concern
Intent: Hello, I placed an order over two weeks ago (Order number #84920) and I have not received any shipping updates yet. Also, the amount has already been charged to my card. I tried calling but no one answers. Please, I need an immediate solution or a refund.
Summary:
Customer has not received any shipping updates for order #84920 placed over two weeks ago, amount has been charged to card, unable to reach anyone via phone, requesting immediate solution or refund
